# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.4 - Evaluation

In this notebook, we are going to evaluate the fine-tuned Qwen 2.5 - 7B Instruct model, using the eval dataset created in the first notebook

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

### Get Model Package Group

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name=f"{project_prefix}-{base_model_shortname}"
model_package_version = "1"

mpg = ModelPackageGroup.get(model_package_group_name)
fine_tuned_model_package_group_arn = mpg.model_package_group_arn
print(fine_tuned_model_package_group_arn)

fine_tuned_model_package_arn = f"{mpg.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(fine_tuned_model_package_arn)

bucket_name = sess.default_bucket()
s3_output_path = f"s3://{bucket_name}/{project_prefix}-{base_model_shortname}/eval/".format(bucket_name)
print(s3_output_path)

### Configure LLM-as-Judge Evaluation

We set up an evaluation job using Amazon Nova Pro as the judge model. The job evaluates both the base model and our RLAIF fine-tuned model. Both use a custom "human-like alignment" metric that scores responses on naturalness, tone, and conversational flow against ground truth references. Running identical evaluations on both models lets us quantify the impact of RLAIF fine-tuning on the human-like-alignment metric defined below.

In [ ]:
import json
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

evaluator_model_id = "amazon.nova-pro-v1:0"
eval_dataset = DataSet.get(f"{project_prefix}-eval")

custom_metrics_list = [
    {
        "customMetricDefinition": {
            "name": "human-like-alignment",
            "instructions": (
                "You are an expert at evaluating language model outputs and "
                "determining which responses sound more natural and human-like. "
                "Be extremely critical and look at it very thoroughly.\n\n"
                "You may use the ground truth response as a reference of what "
                "a human-like answer should contain.\n\n"
                "Focus on these aspects when evaluating:\n"
                "- Human-like reasoning and explanations\n"
                "- Emotional intelligence and empathy where relevant\n"
                "- Avoidance of overly rigid or mechanical language\n"
                "- Natural conversation flow\n"
                "- Appropriate level of formality\n\n"
                "Here is the actual task:\n"
                "Task: {{prompt}}\n"
                "Ground Truth Response: {{ground_truth}}\n"
                "Candidate Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "No part of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 0
                    }
                },
                {
                    "definition": "Roughly half of the response is a human-like response.",
                    "value": {
                        "floatValue": 1
                    }
                },
                {
                    "definition": "Every piece of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 2
                    }
                }
            ]
        }
    }
]

custom_metrics_json = json.dumps(custom_metrics_list)

evaluator = LLMAsJudgeEvaluator(
    model=fine_tuned_model_package_arn,
    model_package_group=fine_tuned_model_package_group_arn,
    evaluator_model=evaluator_model_id,
    dataset=eval_dataset.arn,
    custom_metrics=custom_metrics_json,
    s3_output_path=s3_output_path,  # Required
    evaluate_base_model=True # This will automatically trigger another evaluation, using the base (uncustomized) model
)

### Task 1: Add a Correctness Metric 

Model customization involves multivariate optimization, where improving one dimension can inadvertently impact another. For example, optimizing for human-like alignment may come at the cost of response correctness. Your objective is to define a new metric that validates whether correctness is preserved, and add it to the `custom_metrics_list` metrics list above. Be sure to include a `ratingScale` that enables meaningful comparison between models based on the defined metrics. For a hint, refer to the Task 1 Solution section at the end of this notebook.

### Task 2 (Homework): Use MMLU Dataset for Additional Evaluation 
MMLU (Massive Multitask Language Understanding) is a widely used benchmark for evaluating the knowledge and reasoning capabilities of large language models across 57 diverse subjects, spanning STEM, humanities, social sciences, and professional domains such as law and medicine. The benchmark consists of over 15,000 multiple-choice questions, ranging from elementary to advanced professional difficulty, and supports both zero-shot and few-shot evaluation. MMLU has become a de facto standard for measuring how well a model generalizes across a broad spectrum of human knowledge, making it one of the most referenced benchmarks in LLM development and comparison. Create a slice of the dataset and use it for evaluation of model accuracy. For a solution, refer to the Task 2 Solution section at the end of this notebook.

### Start the Evaluation job

In [ ]:
execution = evaluator.evaluate()

In [ ]:
You may follow up on evaluation job status using Model Evaluation screen in SageMaker AI Studio. Open SageMaker AI Studio, choose Jobs -> Mondel Evalution
from the left panel. Clicking on the corresponding evaluation jobs opens a screen that shows evaluation job details, steps and results. 
If you prefer to use code, the command below blocks execution until the evaluation job completes. It also provides a rich visual experience within Jupyter notebooks, displaying the overall job status, execution time, and progress of each evaluation step.

In [ ]:
execution.wait(poll=30, timeout=3600)

Calling `show_results` allows you to review the detailed evaluation results.

In [ ]:
execution.show_results()

---

## Analyze Evaluation results
### Download the evaluation output from S3

In [ ]:
import os

execution_id = execution.arn.split("/")[-1]
output_path = execution.s3_output_path

print(f"Execution ID: {execution_id}")
print(f"S3 output path: {output_path}")

os.makedirs("./eval_results", exist_ok=True)

s3 = boto3.client("s3", region_name=boto3.Session().region_name)

bucket = output_path.replace("s3://", "").split("/")[0]
prefix = "/".join(output_path.replace("s3://", "").split("/")[1:])

base_key = None
custom_key = None

for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix)["Contents"]:
    key = obj["Key"]
    if execution_id in key and "_output.jsonl" in key and "/datasets/" in key:
        if "base-llmaj-eval" in key:
            base_key = key
        elif "custom-llmaj-eval" in key:
            custom_key = key

print(f"Base: {base_key}")
print(f"Custom: {custom_key}")

s3.download_file(bucket, base_key, "./eval_results/base_eval.jsonl")
s3.download_file(bucket, custom_key, "./eval_results/custom_eval.jsonl")

print("Downloaded to ./eval_results/")


### Parse and display the evaluation results

In [ ]:
# Analyze results
def get_score(result):
    scores = result.get('automatedEvaluationResult', {}).get('scores', [])
    for score in scores:
        if score.get('metricName') == 'human-like-alignment':
            return score.get('result')
    return None

with open('./eval_results/base_eval.jsonl') as f:
    base_results = [json.loads(line) for line in f]

with open('./eval_results/custom_eval.jsonl') as f:
    custom_results = [json.loads(line) for line in f]

base_scores = [get_score(r) for r in base_results if get_score(r) is not None]
custom_scores = [get_score(r) for r in custom_results if get_score(r) is not None]

# Calculate metrics (scores are 0.0, 1.0, 2.0)
base_avg = sum(base_scores) / len(base_scores) / 2 * 100 if base_scores else 0
custom_avg = sum(custom_scores) / len(custom_scores) / 2 * 100 if custom_scores else 0

# Distribution
def get_distribution(scores):
    completely = sum(1 for s in scores if s == 2.0)
    somewhat = sum(1 for s in scores if s == 1.0)
    not_at_all = sum(1 for s in scores if s == 0.0)
    return completely, somewhat, not_at_all

base_dist = get_distribution(base_scores)
custom_dist = get_distribution(custom_scores)

print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)
print(f"\nBase Model:")
print(f"  Average Score: {base_avg:.1f}%")
print(f"  Valid Samples: {len(base_scores)}/{len(base_results)}")
print(f"  Distribution:")
print(f"    - Completely human-like (2.0): {base_dist[0]} ({base_dist[0]/len(base_scores)*100:.1f}%)")
print(f"    - Somewhat human-like (1.0): {base_dist[1]} ({base_dist[1]/len(base_scores)*100:.1f}%)")
print(f"    - Not at all human-like (0.0): {base_dist[2]} ({base_dist[2]/len(base_scores)*100:.1f}%)")

print(f"\nFine-tuned Model:")
print(f"  Average Score: {custom_avg:.1f}%")
print(f"  Valid Samples: {len(custom_scores)}/{len(custom_results)}")
print(f"  Distribution:")
print(f"    - Completely human-like (2.0): {custom_dist[0]} ({custom_dist[0]/len(custom_scores)*100:.1f}%)")
print(f"    - Somewhat human-like (1.0): {custom_dist[1]} ({custom_dist[1]/len(custom_scores)*100:.1f}%)")
print(f"    - Not at all human-like (0.0): {custom_dist[2]} ({custom_dist[2]/len(custom_scores)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"IMPROVEMENT: {custom_avg - base_avg:+.1f}%")
print(f"{'='*80}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Distribution comparison
categories = ['Not at all\n(0.0)', 'Somewhat\n(1.0)', 'Completely\n(2.0)']
base_counts = [base_dist[2], base_dist[1], base_dist[0]]
custom_counts = [custom_dist[2], custom_dist[1], custom_dist[0]]

x = np.arange(len(categories))
width = 0.35

bars1 = ax1.bar(x - width/2, base_counts, width, label='Base Model', alpha=0.8)
bars2 = ax1.bar(x + width/2, custom_counts, width, label='Fine-tuned Model', alpha=0.8)

ax1.set_xlabel('Human-like Alignment Score')
ax1.set_ylabel('Number of Samples')
ax1.set_title('Distribution of Scores')
ax1.set_xticks(x)
ax1.set_xticklabels(categories)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Average score comparison
models = ['Base Model', 'Fine-tuned Model']
averages = [base_avg, custom_avg]
colors = ['#1f77b4', '#ff7f0e']

bars = ax2.bar(models, averages, color=colors, alpha=0.8)
ax2.set_ylabel('Average Score (%)')
ax2.set_title('Average Human-like Alignment Score')
ax2.set_ylim(0, 100)
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Task 1 Solution

In [ ]:
custom_metrics_list = [
    {
        "customMetricDefinition": {
            "name": "human-like-alignment",
            "instructions": (
                "You are an expert at evaluating language model outputs and "
                "determining which responses sound more natural and human-like. "
                "Be extremely critical and look at it very thoroughly.\n\n"
                "You may use the ground truth response as a reference of what "
                "a human-like answer should contain.\n\n"
                "Focus on these aspects when evaluating:\n"
                "- Human-like reasoning and explanations\n"
                "- Emotional intelligence and empathy where relevant\n"
                "- Avoidance of overly rigid or mechanical language\n"
                "- Natural conversation flow\n"
                "- Appropriate level of formality\n\n"
                "Here is the actual task:\n"
                "Task: {{prompt}}\n"
                "Ground Truth Response: {{ground_truth}}\n"
                "Candidate Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "No part of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 0
                    }
                },
                {
                    "definition": "Roughly half of the response is a human-like response.",
                    "value": {
                        "floatValue": 1
                    }
                },
                {
                    "definition": "Every piece of the response sounds like a human-like response.",
                    "value": {
                        "floatValue": 2
                    }
                }
            ]
        }
    },
    {
        "customMetricDefinition": {
            "name": "correctness",
            "instructions": (
                "Rate the correctness of the answer on a scale of 0.0 to 1.0, where 0.0 is completely wrong and 1.0 is completely correct.\n\n"
                "Consider factual accuracy and completeness. In result, report a floating point number indicating correctness only.\n\n"
                "Be extremely critical and look at it very thoroughly.\n\n"
                "You may use the ground truth response as a reference of what "
                "a correct answer should contain.\n\n"
                "Here is the actual task:\n"
                "Task: {{prompt}}\n"
                "Ground Truth Response: {{ground_truth}}\n"
                "Candidate Response: {{prediction}}"
            ),
            "ratingScale": [
                {
                    "definition": "Correctness is below 0.2",
                    "value": {
                        "floatValue": 0
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.2 and below 0.3",
                    "value": {
                        "floatValue": 0.2
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.3 and below 0.4",
                    "value": {
                        "floatValue": 0.3
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.4 and below 0.5",
                    "value": {
                        "floatValue": 0.4
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.5 and below 0.6",
                    "value": {
                        "floatValue": 0.5
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.6 and below 0.7",
                    "value": {
                        "floatValue": 0.6
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.7 and below 0.8",
                    "value": {
                        "floatValue": 0.7
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.8 and below 0.9",
                    "value": {
                        "floatValue": 0.8
                    }
                },
                {
                    "definition": "Correctness is equal or above 0.9",
                    "value": {
                        "floatValue": 1.0
                    }
                }
            ]
        }
    }
]

## Task 2 Solution 
### Prepare an MMLU dataset slice

Download 100 entries from MMLU dataset and store them in JSONL format for evaluation with SageMaker.

In [ ]:
import pandas as pd
import os
from datasets import load_dataset
from sagemaker.ai_registry.dataset import DataSet

# Create a dataset from MMLU

NUM_SAMPLES = 1000

ds = load_dataset("cais/mmlu", "all", split="test")
ds = ds.shuffle(seed=42).select(range(NUM_SAMPLES))
df = ds.to_pandas()

CHOICE_LABELS = ['A', 'B', 'C', 'D']

df["subject_clean"] = df["subject"].fillna("general knowledge").str.replace("_", " ")
df["correct_text"] = df.apply(lambda r: r["choices"][r["answer"]], axis=1)
df["correct_label"] = df["answer"].map(dict(enumerate(CHOICE_LABELS)))
df["choices_str"] = df["choices"].apply(
    lambda cs: '\n'.join(f"{CHOICE_LABELS[i]}. {c}" for i, c in enumerate(cs))
)

df["query"] = df.apply(lambda r: (
    f"You are an expert in {r['subject_clean']}. Answer the following question.\n"
    f"Instructions:\n"
    f"- Provide a clear, concise answer in 2-3 sentences.\n"
    f"- Explain your reasoning briefly.\n"
    f"- State the subject area at the end of your response in brackets.\n\n"
    f"Question: {r['question']}\n\n"
    f"Choices:\n{r['choices_str']}"
), axis=1)

df["response"] = df.apply(
    lambda r: f"{r['correct_label']}. {r['correct_text']}. [{r['subject_clean']}]", axis=1
)
df[["query", "response"]].to_json(f"./datasets/{project_prefix}-mmlu-eval.jsonl", orient="records", lines=True)


# Download to S3
local_path = "./datasets/"
file_name = f"{project_prefix}-mmlu-eval.jsonl"
prefix_key = f"{project_prefix}/{file_name}"
s3_client.upload_file(os.path.join(local_path, file_name), bucket_name, prefix_key)

eval_data_uri = f"s3://{bucket_name}/{prefix_key}"
print(eval_data_uri)

# Register in datasets registry
eval_dataset = DataSet.create(
    name=f"{project_prefix}-mmlu-eval",
    source=eval_data_uri,
    sagemaker_session=sess,
    wait=True
)

print(f"Evaluation dataset ARN: {eval_dataset.arn}")
eval_dataset_arn = eval_dataset.arn

### Define LLM-as-Judge evaluator

We set up an evaluation job using Amazon Nova as the judge model while using a new dataset and a new metric defined in the previous task.

In [ ]:
custom_metrics_json = json.dumps(custom_metrics_list)

evaluator = LLMAsJudgeEvaluator(
    base_model=base_model_jumpstart_id,  # JumpStart ID
    model=fine_tuned_model_package_arn,
    evaluator_model=evaluator_model_id,
    dataset=eval_dataset_arn,
    custom_metrics=custom_metrics_json,
    s3_output_path=s3_output_path,  # Required
    evaluate_base_model=True
)

### Start evaluation job
The code below starts evaluation job, waits for evaluation completion and shows the evaluation results. 

In [ ]:
execution = evaluator.evaluate()
print(f"Evaluation job ARN: {execution.arn}")

execution.wait(poll=30, timeout=3600)
execution.show_results()